# 🔬 Notebook 4: Downstream Single-Cell Analysis (Scanpy, UMAP, Leiden & Markers)

This notebook walks through the complete downstream single-cell analysis workflow:
1. **Quality Control & Filtering**: Cell & gene filtering, mitochondrial percentage thresholds
2. **Normalization & Log1p**: Library size scaling & variance stabilization
3. **Highly Variable Genes (HVGs)**: Feature selection
4. **Dimensionality Reduction**: Principal Component Analysis (PCA) & Neighborhood Graph
5. **Non-Linear Embedding**: UMAP visualization
6. **Cell Clustering**: Leiden community detection algorithm
7. **Marker Gene Discovery**: Cluster-specific differential expression (Wilcoxon rank-sum test)
8. **Visualization**: Publication-quality UMAPs, DotPlots, and Heatmaps

In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import scanpy as sc
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import load_config
from src.downstream import SingleCellDownstream

sc.set_figure_params(dpi=120, facecolor='white', frameon=True)
config = load_config('../config/pipeline_config.yaml')

### 1. Load Count Matrix & Compute QC Metrics

In [ ]:
ds = SingleCellDownstream(config)
adata = ds.load_anndata()
adata = ds.run_qc_and_filtering(adata)
adata

### 2. Normalization & HVG Selection

In [ ]:
adata = ds.run_normalization_and_hvg(adata)
print(f"Highly variable genes: {adata.var['highly_variable'].sum()} / {adata.n_vars}")

### 3. PCA, Neighborhood Graph, UMAP & Leiden Clustering

In [ ]:
adata = ds.run_clustering_and_embeddings(adata)
sc.pl.umap(adata, color=['leiden', 'total_counts', 'n_genes_by_counts'], ncols=3)

### 4. Cluster Marker Gene Discovery (Differential Expression)

In [ ]:
df_markers = ds.run_marker_discovery(adata)
if not df_markers.empty:
    display(df_markers.head(10))
    top_genes = list(df_markers.groupby('cluster')['gene'].first().unique())
    sc.pl.dotplot(adata, var_names=top_genes, groupby='leiden')